In [9]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import torch.optim as optim
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# data

In [ ]:
# Define the custom color palette (Rust-Red, Orange, Gold, Tan)
nfl_colors = ['#a83216', '#d8631d', '#e29726', '#b29d83']

# Apply the custom color cycle and palette globally
sns.set_palette(sns.color_palette(nfl_colors))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=nfl_colors)

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}


train_df = pd.read_parquet("hf://datasets/afzalmengal/nfl-4th-downs-dataset/" + splits["train"])
train_df['play_type'] = train_df['play_type'].replace({'pass': 'go', 'run': 'go', 'qb_kneel' : 'no_play'})

val_df = pd.read_parquet("hf://datasets/afzalmengal/nfl-4th-downs-dataset/" + splits["validation"])
val_df['play_type'] = val_df['play_type'].replace({'pass': 'go', 'run': 'go', 'qb_kneel' : 'no_play'})




<ArrowStringArray>
['punt', 'field_goal', 'pass', 'no_play', nan, 'run', 'qb_kneel']
Length: 7, dtype: str

In [ ]:
continuous_features = [
    'score_differential', 'ydstogo', 'yardline_100', 'game_seconds_remaining', 
    'spread_line', 'total_line', 'wp', 'SRS Rating', 'SOS Rating', 
    'Points For Per Game', 'Points Against Per Game'
]
categorical_features = ['posteam', 'defteam', 'posteam_type', 'goal_to_go', 'qtr', 'season_type']
target_action = 'play_type'
target_wpa = 'wpa'


X_cat = train_df[categorical_features]
X_cont = train_df[continuous_features]

In [6]:
train_df.columns

Index(['game_id', 'season', 'home_team', 'away_team', 'posteam', 'defteam',
       'qtr', 'game_seconds_remaining', 'yardline_100', 'ydstogo',
       'goal_to_go', 'score_differential', 'home_score', 'away_score', 'wp',
       'def_wp', 'spread_line', 'total_line', 'play_type', 'posteam_type',
       'home_coach', 'away_coach', 'epa', 'wpa', 'success',
       'fourth_down_converted', 'fourth_down_failed', 'label', 'week',
       'season_type', 'Division', 'wins', 'losses', 'ties', 'league_rank',
       'division_rank', 'wins_yearly', 'losses_yearly', 'ties_yearly',
       'Division Placement', 'Division Name', 'Points For',
       'Points For Per Game', 'Points For Ranking', 'Points Against',
       'Points Against Per Game', 'Points Against Ranking', 'Expected Wins',
       'Expected Losses', 'SRS', 'SRS Rank', 'SOS', 'SRS Rating', 'SOS Rating',
       '__index_level_0__', 'game_fraction_remaining'],
      dtype='str')

In [16]:
from models.embedding_model import Embedding_Model
from models.lstm import embedded_LSTM



In [15]:
from utils.dataprocessing import data_to_tensors


X_cat_tensor = torch.tensor(X_cat.values)
X_cont_tensor = torch.tensor(X_cont.values)
A_tensor = torch.tensor(train_df[target_action].values)
Y_tensor = torch.tensor(train_df[target_wpa].values)


# =====================================================================
# 3. EXECUTION EXAMPLE
# =====================================================================
if __name__ == "__main__":
    # 1. Configuration Dictionary
    config = {
        'cat_emb_dims': [(32, 16), (2, 2), (5, 3)], # Example: (Unique teams, embedding size)
        'num_continuous': 11, # Number of continuous features in your dataset
        'hidden_size': 64,
        'learning_rate': 0.001,
        'epochs': 5,
        'batch_size': 64,
        'pretrained_embedding_path': None # Set to a .pth file to load and freeze embeddings
    }
    
    # 2. Initialize the wrapper class
    model = embedded_LSTM(config)
    
    # 3. Dummy Data (Replace this with your extracted dataset tensors)
    # X needs to be a dictionary separating the continuous and categorical variables
    dummy_X = {
        'cont': X_cat_tensor,       # 128 sequences, 4 plays long, 11 continuous features
        'cat': X_cont_tensor # 128 sequences, 4 plays long, 3 categorical features
    }

    
    # 4. Train the Model
    model.train(dummy_X, A_tensor, Y_tensor)
    
    # 5. Predict using the Model
    pred_actions, pred_wpa = model.predict(dummy_X)
    print("\nSample Predictions (Play Type):", pred_actions[:5])
    print("Sample Predictions (WPA):", pred_wpa[:5])

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.